In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt


2026-02-13 05:22:25.219160: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770960145.422070      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770960145.483011      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770960145.948527      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770960145.948598      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770960145.948603      55 computation_placer.cc:177] computation placer alr

In [2]:
print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [6]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("  ", filename)


/kaggle/input
/kaggle/input/imdb-dataset
   IMDB Dataset.csv


In [8]:
import pandas as pd

df = pd.read_csv("/kaggle/input/imdb-dataset/IMDB Dataset.csv")

print(df.head())
print(df.shape)


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [9]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

texts = df['review'].values
labels = df['sentiment'].values


In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
max_length = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post'
)


In [11]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    padded_sequences,
    labels,
    test_size=0.2,
    random_state=42
)


In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Embedding(vocab_size, 128, input_length=max_length),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-02-13 05:35:26.920833: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)


Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 62ms/step - accuracy: 0.6750 - loss: 0.5557 - val_accuracy: 0.8662 - val_loss: 0.3191
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 0.9638 - loss: 0.1059 - val_accuracy: 0.8528 - val_loss: 0.4051
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 0.9974 - loss: 0.0121 - val_accuracy: 0.8611 - val_loss: 0.5842
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 0.9997 - loss: 0.0016 - val_accuracy: 0.8581 - val_loss: 0.7123
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 59ms/step - accuracy: 1.0000 - loss: 1.2980e-04 - val_accuracy: 0.8627 - val_loss: 0.7548


In [14]:
loss, accuracy = model.evaluate(x_test, y_test)
print("Test Accuracy:", accuracy)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8596 - loss: 0.7558
Test Accuracy: 0.8615000247955322


In [15]:
sample_review = ["The movie was boring and a waste of time"]

seq = tokenizer.texts_to_sequences(sample_review)
pad = pad_sequences(seq, maxlen=max_length, padding='post')

prediction = model.predict(pad)

print("Positive Review" if prediction[0][0] > 0.5 else "Negative Review")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
Negative Review
